<a href="https://colab.research.google.com/github/Sarkis55/UHC-and-Population-Health-Outcomes-Analysis-SDG-3-/blob/main/UHC_and_Population_Health_Outcomes_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Does Universal Health Care Coverage Improve Population Health Outcome?

To what extent does the UHC Service Coverage Index (SDG 3.8.1) predict differences in key population health outcomes across countries? This analysis will test whether higher UHC coverage levels are associated with lower maternal mortality (3.1.1), number of HIV infections (3.3.1), and decreased premature mortality from noncommunicable diseases (3.4.1). The study will compare countries across regions and income groups to assess the strength, direction, and consistency of these relationships. Findings will clarify whether improvements in health coverage systematically align with better national health outcomes.



------

------

### Loading the Data, through SDG API, into CSV files to create dataframes





In [1]:
import pandas as pd
import requests
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

In [2]:
#Loading entire SDG Data via the UNSD SDG API
series_url = "https://unstats.un.org/SDGAPI/v1/sdg/Series/List"
series = requests.get(series_url).json()
series_df = pd.DataFrame(series)
series_df.head()

,goal,target,indicator,release,code,description,uri
0,"[15, 15]","[15.a, 15.b]","[15.a.1, 15.b.1]",2026.Q1.G.01,DC_ODA_BDVDL,Total official development assistance for biod...,/v1/sdg/Series/DC_ODA_BDVDL
1,[3],[3.b],[3.b.2],2026.Q1.G.01,DC_TOF_HLTHNT,Total official development assistance to medic...,/v1/sdg/Series/DC_TOF_HLTHNT
2,[9],[9.a],[9.a.1],2026.Q1.G.01,DC_TOF_INFRAL,"Total official flows for infrastructure, by re...",/v1/sdg/Series/DC_TOF_INFRAL
3,[4],[4.b],[4.b.1],2026.Q1.G.01,DC_TOF_SCHIPSL,"Total official flows for scholarships, by reci...",/v1/sdg/Series/DC_TOF_SCHIPSL
4,[8],[8.a],[8.a.1],2026.Q1.G.01,DC_TOF_TRDCMDL,Total official flows (commitments) for Aid for...,/v1/sdg/Series/DC_TOF_TRDCMDL


In [3]:
#Since we are only concerned with Goal #3, Good Health and Well-Being,
#we filter down to the series relating to health.
#For Goal 3: Good Health and Wellbeing, all code start with an 'SH_'

health_series = series_df[series_df['code'].str.startswith("SH_", na = False)].copy()
health_series[['code', 'description']].head(100)

,code,description
34,SH_DTH_NCOM,Mortality rate attributed to cardiovascular di...
35,SH_DYN_IMRTN,Infant deaths (number)
36,SH_DYN_MORT,"Under-five mortality rate, by sex (deaths per ..."
37,SH_DYN_NMRTN,Neonatal deaths (number)
38,SH_H2O_SAFE,Proportion of population using safely managed ...
...,...,...
584,SH_STA_WASHARI,"Mortality rate attributed to unsafe water, uns..."
674,SH_MDD_WMN_NPRG,Prevalence of minimum dietary diversity among ...
687,SH_MDD_CHLD,Prevalence of minimum dietary diversity among ...
688,SH_OOP_XPD_EARNNET40,Proportion of the population with positive out...


In [4]:
#Function that takes the SDG API and returns back the contents of the desired series

def get_content(url, data_dict):
  r = requests.post(
      url,
      headers = {
          "Accept": "application-octet-stream",
          "Content-Type": "application/x-www-form-urlencoded",
      },
     data = data_dict
  )

  print("Status: ", r.status_code)
  print("Bytes: ", len(r.content))
  print("Content-Type: ", r.headers.get('content-type'))

  return r.content

In [5]:
#We then load the desired series into their own respective dictionaries
url = "https://unstats.un.org/sdgapi/v1/sdg/Series/DataCSV"

UNHC_dicts= {'seriesCodes': "SH_ACS_UNHC_25"}
TreatDrug_dict = {'seriesCodes': "SH_SUD_TREAT"}
HIV_dict = {'seriesCodes': "SH_HIV_INCD"}
Diseases_dict = {'seriesCodes': "SH_DTH_NCOM"}

#-----------------------------------------------------------------------------------------------

#Then we run each dictionary and url of the desired series through the get_content function, and get returned back a CSV file
UNHC_data = get_content(url, UNHC_dicts)
with open("SH_ACS_UNHC_25.csv", "wb") as f:
    f.write(UNHC_data)

Drug_Treatment_data = get_content(url, TreatDrug_dict)
with open("SH_SUD_TREAT.csv", "wb") as f:
    f.write(Drug_Treatment_data)

HIV_data = get_content(url, HIV_dict)
with open("SH_HIV_INCD.csv", "wb") as f:
    f.write(HIV_data)

Diseases_data = get_content(url, Diseases_dict)
with open("SH_DTH_NCOM.csv", "wb") as f:
    f.write(Diseases_data)

Status:  200
Bytes:  2097152
Content-Type:  application/octet-stream
Status:  200
Bytes:  1048576
Content-Type:  application/octet-stream
Status:  200
Bytes:  33554432
Content-Type:  application/octet-stream
Status:  200
Bytes:  4194304
Content-Type:  application/octet-stream




---
---
# Data Preprocessing



---
---
## SDG 3.8.1: Universal Health Care Services for each Country (2000-2023)

Coverage of essential health services (defined as the average coverage of essential services based on tracer intervention).

The tracer indicators are as follows, organized by four subindices of service coverage:

1. Reproductive, maternal, newborn and child health

2. Infectious diseases

3. Noncommunicable diseases

4. Service capacity and access

The index `value` is reported on a unitless scale of 0 to 100, where 100 is the target "optimal" value. Rather than measuring a single health metric, it is a composite index calculated as the geometric mean of 14 tracer indicators of the four main categories. **Essentially, the higher the number, the better a country is at providing essential health services to its entire population without coverage gaps.**


In [24]:
#Creating a dataframe for SDG 3.8.1, Universal Health Care Coverage for each Country
UNHC_df = pd.read_csv("SH_ACS_UNHC_25.csv")
UNHC_df.head()

,Goal,Target,Indicator,SeriesCode,SeriesDescription,GeoAreaCode,GeoAreaName,TimePeriod,Value,Time_Detail,TimeCoverage,UpperBound,LowerBound,BasePeriod,Source,GeoInfoUrl,FootNote,[Nature],[Reporting Type],[Units]
0,3.0,3.8,3.8.1,SH_ACS_UNHC_25,Universal health coverage (UHC) service covera...,96.0,Brunei Darussalam,2014.0,84.0,2014.0,NaN,NaN,NaN,NaN,"WHO Global Health Observatory (GHO),December 2...",NaN,NaN,E,G,INDEX
1,3.0,3.8,3.8.1,SH_ACS_UNHC_25,Universal health coverage (UHC) service covera...,96.0,Brunei Darussalam,2015.0,84.0,2015.0,NaN,NaN,NaN,NaN,"WHO Global Health Observatory (GHO),December 2...",NaN,NaN,E,G,INDEX
2,3.0,3.8,3.8.1,SH_ACS_UNHC_25,Universal health coverage (UHC) service covera...,96.0,Brunei Darussalam,2016.0,83.0,2016.0,NaN,NaN,NaN,NaN,"WHO Global Health Observatory (GHO),December 2...",NaN,NaN,E,G,INDEX
3,3.0,3.8,3.8.1,SH_ACS_UNHC_25,Universal health coverage (UHC) service covera...,96.0,Brunei Darussalam,2017.0,83.0,2017.0,NaN,NaN,NaN,NaN,"WHO Global Health Observatory (GHO),December 2...",NaN,NaN,E,G,INDEX
4,3.0,3.8,3.8.1,SH_ACS_UNHC_25,Universal health coverage (UHC) service covera...,96.0,Brunei Darussalam,2018.0,83.0,2018.0,NaN,NaN,NaN,NaN,"WHO Global Health Observatory (GHO),December 2...",NaN,NaN,E,G,INDEX


In [26]:
UNHC_df.shape

(5185, 20)

In [27]:
'''
[Nature] is the source of the datapoints.
    - N represents National, where the data was officially reported by the country
    - E represents Estimated, where the data was not reported and is fill in by an international agency
'''

UNHC_df['[Nature]'].value_counts()

,count
[Nature],
E,4680
N,504


In [28]:
#Checking for missing values
UNHC_df.isna().sum()

,0
Goal,1
Target,1
Indicator,1
SeriesCode,1
SeriesDescription,1
GeoAreaCode,1
GeoAreaName,1
TimePeriod,1
Value,1
Time_Detail,1


In [29]:
print(UNHC_df['TimePeriod'].corr(UNHC_df['Time_Detail']))

#Dropping Time_Detail becaus it is the same as TimePeriod
UNHC_df.drop('Time_Detail', axis = 1, inplace = True)

1.0


In [30]:
#Dropping columns that are full of NA value and with a cardinality of 1.
UNHC_df.drop(["Goal",'Target', 'Indicator', "SeriesCode", "SeriesDescription", "TimeCoverage", 'UpperBound', 'LowerBound', 'BasePeriod', 'Source', 'GeoInfoUrl','FootNote', '[Reporting Type]', '[Units]'], axis = 1, inplace = True)

#Dropping rows with null values, since there is only 1
UNHC_df.dropna(inplace = True)

,GeoAreaCode,GeoAreaName,TimePeriod,Value,[Nature]
0,96.0,Brunei Darussalam,2014.0,84.0,E
1,96.0,Brunei Darussalam,2015.0,84.0,E
2,96.0,Brunei Darussalam,2016.0,83.0,E
3,96.0,Brunei Darussalam,2017.0,83.0,E
4,96.0,Brunei Darussalam,2018.0,83.0,E


In [31]:
#Normalizing the Value column so that it can be analzyed and compared with value columns of other datasets
UNHC_df[['Value']] = scaler.fit_transform(UNHC_df[['Value']])

In [32]:
UNHC_df.head()

,GeoAreaCode,GeoAreaName,TimePeriod,Value,[Nature]
0,96.0,Brunei Darussalam,2014.0,0.898734,E
1,96.0,Brunei Darussalam,2015.0,0.898734,E
2,96.0,Brunei Darussalam,2016.0,0.886076,E
3,96.0,Brunei Darussalam,2017.0,0.886076,E
4,96.0,Brunei Darussalam,2018.0,0.886076,E


---
---
## SDG 3.3.1 Number of New HIV Infections

Measures the number of new HIV infections per 1,000 uninfected population.

In the UNSDG database, this `value` represents the incidence rate, which tracks how fast the virus is spreading within a specific country or region. Unlike "prevalence" (which counts everyone currently living with HIV), incidence focuses only on brand-new cases identified during the reporting year.

Key Details for Analysis:

- Unit of Measurement: The value is expressed as the number of new cases per 1,000 uninfected people. For example, a value of
0.15 means that for every 1,000 people who did not have HIV at the start of the year, 0.15 people (or roughly 15 out of 100,000) became infected.
- Target: This is a primary metric for SDG Target 3.3, which aims to end the epidemics of AIDS, tuberculosis, malaria, and neglected tropical diseases by 2030.
- Demographic Breakdowns: This dataset is often broken down by sex (male/female) and age (typically 15–49 or 15–24), as these breakdowns are crucial for identifying which specific groups are at the highest risk.


In [62]:
#Creating a Dataframe for SDG 3.3.1, HIV Infected count by thousands
HIV_df = pd.read_csv("SH_HIV_INCD.csv")
HIV_df.head()

/tmp/ipykernel_21235/2794687962.py:2: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  HIV_df = pd.read_csv("SH_HIV_INCD.csv")


,Goal,Target,Indicator,SeriesCode,SeriesDescription,GeoAreaCode,GeoAreaName,TimePeriod,Value,Time_Detail,...,BasePeriod,Source,GeoInfoUrl,FootNote,[Age],[Nature],[Observation Status],[Reporting Type],[Sex],[Units]
0,3.0,3.3,3.3.1,SH_HIV_INCD,"Number of new HIV infections per 1,000 uninfec...",1.0,World,2000.0,0.66052,2000.0,...,NaN,UNAIDS 2025 HIV ESTIMATES,NaN,NaN,15-49,N,E,G,MALE,PER_1000_UNINFECTED_POP
1,3.0,3.3,3.3.1,SH_HIV_INCD,"Number of new HIV infections per 1,000 uninfec...",1.0,World,2000.0,0.14628,2000.0,...,NaN,UNAIDS 2025 HIV ESTIMATES,NaN,NaN,50+,N,E,G,BOTHSEX,PER_1000_UNINFECTED_POP
2,3.0,3.3,3.3.1,SH_HIV_INCD,"Number of new HIV infections per 1,000 uninfec...",1.0,World,2000.0,0.46730,2000.0,...,NaN,UNAIDS 2025 HIV ESTIMATES,NaN,NaN,ALLAGE,N,E,G,MALE,PER_1000_UNINFECTED_POP
3,3.0,3.3,3.3.1,SH_HIV_INCD,"Number of new HIV infections per 1,000 uninfec...",1.0,World,2000.0,0.65706,2000.0,...,NaN,UNAIDS 2025 HIV ESTIMATES,NaN,NaN,15-24,N,E,G,MALE,PER_1000_UNINFECTED_POP
4,3.0,3.3,3.3.1,SH_HIV_INCD,"Number of new HIV infections per 1,000 uninfec...",1.0,World,2000.0,0.50110,2000.0,...,NaN,UNAIDS 2025 HIV ESTIMATES,NaN,NaN,ALLAGE,N,E,G,FEMALE,PER_1000_UNINFECTED_POP


In [63]:
HIV_df.isna().sum()

,0
Goal,1
Target,1
Indicator,1
SeriesCode,1
SeriesDescription,1
GeoAreaCode,1
GeoAreaName,1
TimePeriod,1
Value,15433
Time_Detail,1


In [64]:
HIV_df.drop(["Goal",'Target', 'Indicator', "SeriesCode", "SeriesDescription", "TimeCoverage", 'Time_Detail','UpperBound', 'LowerBound', 'BasePeriod', 'Source', 'GeoInfoUrl','FootNote', '[Reporting Type]',"[Observation Status]",'[Units]'], axis = 1, inplace = True)

In [65]:
HIV_df.dropna(inplace = True)
HIV_df.head(10)

,GeoAreaCode,GeoAreaName,TimePeriod,Value,[Age],[Nature],[Sex]
0,1.0,World,2000.0,0.66052,15-49,N,MALE
1,1.0,World,2000.0,0.14628,50+,N,BOTHSEX
2,1.0,World,2000.0,0.46730,ALLAGE,N,MALE
3,1.0,World,2000.0,0.65706,15-24,N,MALE
4,1.0,World,2000.0,0.50110,ALLAGE,N,FEMALE
5,1.0,World,2000.0,0.16525,50+,N,MALE
6,1.0,World,2000.0,0.12946,50+,N,FEMALE
7,1.0,World,2000.0,0.87300,15-24,N,BOTHSEX
8,1.0,World,2000.0,0.74618,15-49,N,FEMALE
9,1.0,World,2000.0,1.09966,15-24,N,FEMALE


In [66]:
HIV_df.shape

(50868, 7)

In [67]:
HIV_df["[Age]"].value_counts()

,count
[Age],
ALLAGE,12220
15-49,12139
50+,12090
15-24,11898
<15Y,2521


In [68]:
HIV_df["[Nature]"].value_counts()

,count
[Nature],
E,44193
N,6675


In [69]:
'''
For this scenerio, we are comparing countrywide results as a whole so we care more about age and gender as a population in total,
therefore we will filter out the Sex column for just the "BOTHSEX" value and Age column for just "ALLAGE"
'''

Total_HIV_df = HIV_df[(HIV_df["[Sex]"] == "BOTHSEX") & (HIV_df["[Age]"] == "ALLAGE")]

In [70]:
#Normallizing the Value columns
Total_HIV_df["Value"] = scaler.fit_transform(Total_HIV_df[["Value"]])

#Then we flip the Value that way a high value is a good thing and a low value is the bad
Total_HIV_df.loc['Value'] = 1 - Total_HIV_df['Value']

/tmp/ipykernel_21235/3179443194.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Total_HIV_df["Value"] = scaler.fit_transform(Total_HIV_df[["Value"]])
/tmp/ipykernel_21235/3179443194.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Total_HIV_df.loc['Value'] = 1 - Total_HIV_df['Value']


In [71]:
Total_HIV_df.head()

,GeoAreaCode,GeoAreaName,TimePeriod,Value,[Age],[Nature],[Sex]
11,1.0,World,2000.0,0.020232,ALLAGE,N,BOTHSEX
20,1.0,World,2001.0,0.019263,ALLAGE,N,BOTHSEX
29,1.0,World,2002.0,0.018441,ALLAGE,N,BOTHSEX
39,1.0,World,2003.0,0.017571,ALLAGE,N,BOTHSEX
64,1.0,World,2004.0,0.016974,ALLAGE,N,BOTHSEX


In [72]:
Total_HIV_df.shape

(4091, 7)

---
---
## SDG 3.4.1 Mortality Rate Attributed to Noncommunicable Diseases

Measures the mortality rate attributed to cardiovascular disease, cancer, diabetes, or chronic respiratory disease.

The `value` represents the probability of dying between the exact ages of 30 and 70 from any of these four causes. It is calculated using life table methods and expressed as a percentage. This "premature mortality" focus is used because deaths in this age range are often preventable through public health interventions, such as tobacco control, sodium reduction, and improved access to primary healthcare.

Because this indicator tracks four distinct disease groups, it serves as a broad pulse-check for a nation's overall health system and lifestyle risks. A high value in this series generally indicates a need for stronger policies regarding diet, exercise, and the management of chronic conditions like hypertension.


In [113]:
#Creating a Dataframe for SDG 3.4.1, Mortality rate attributed to diseases
Diseases_df = pd.read_csv("SH_DTH_NCOM.csv")
Diseases_df.head()

,Goal,Target,Indicator,SeriesCode,SeriesDescription,GeoAreaCode,GeoAreaName,TimePeriod,Value,Time_Detail,...,BasePeriod,Source,GeoInfoUrl,FootNote,[Age],[Name of non-communicable disease],[Nature],[Reporting Type],[Sex],[Units]
0,3.0,3.4,3.4.1,SH_DTH_NCOM,Mortality rate attributed to cardiovascular di...,1.0,World,2000.0,18.3,2000.0,...,NaN,"Global Health Estimates 2021: Deaths by Cause,...",NaN,Data was previously disseminated with a differ...,30-69,NaN,N,G,FEMALE,PERCENT
1,3.0,3.4,3.4.1,SH_DTH_NCOM,Mortality rate attributed to cardiovascular di...,1.0,World,2000.0,26.8,2000.0,...,NaN,"Global Health Estimates 2021: Deaths by Cause,...",NaN,Data was previously disseminated with a differ...,30-69,NaN,N,G,MALE,PERCENT
2,3.0,3.4,3.4.1,SH_DTH_NCOM,Mortality rate attributed to cardiovascular di...,1.0,World,2000.0,22.5,2000.0,...,NaN,"Global Health Estimates 2021: Deaths by Cause,...",NaN,Data was previously disseminated with a differ...,30-69,NaN,N,G,BOTHSEX,PERCENT
3,3.0,3.4,3.4.1,SH_DTH_NCOM,Mortality rate attributed to cardiovascular di...,1.0,World,2005.0,21.2,2005.0,...,NaN,"Global Health Estimates 2021: Deaths by Cause,...",NaN,Data was previously disseminated with a differ...,30-69,NaN,N,G,BOTHSEX,PERCENT
4,3.0,3.4,3.4.1,SH_DTH_NCOM,Mortality rate attributed to cardiovascular di...,1.0,World,2005.0,25.4,2005.0,...,NaN,"Global Health Estimates 2021: Deaths by Cause,...",NaN,Data was previously disseminated with a differ...,30-69,NaN,N,G,MALE,PERCENT


In [114]:
Diseases_df.isna().sum()

,0
Goal,1
Target,1
Indicator,1
SeriesCode,1
SeriesDescription,1
GeoAreaCode,1
GeoAreaName,1
TimePeriod,1
Value,1
Time_Detail,1


In [115]:
Diseases_df['[Age]'].value_counts()

,count
[Age],
30-69,4326


In [116]:
Diseases_df.drop(["Goal",'Target', 'Indicator', "SeriesCode", "SeriesDescription", "TimeCoverage", 'Time_Detail','BasePeriod', 'Source', 'GeoInfoUrl','FootNote','[Name of non-communicable disease]', '[Age]','[Reporting Type]','[Units]'], axis = 1, inplace = True)

In [117]:
Diseases_df.head()

,GeoAreaCode,GeoAreaName,TimePeriod,Value,UpperBound,LowerBound,[Nature],[Sex]
0,1.0,World,2000.0,18.3,22.3,14.7,N,FEMALE
1,1.0,World,2000.0,26.8,31.7,22.6,N,MALE
2,1.0,World,2000.0,22.5,27.0,18.6,N,BOTHSEX
3,1.0,World,2005.0,21.2,25.4,17.7,N,BOTHSEX
4,1.0,World,2005.0,25.4,30.1,21.6,N,MALE


In [118]:
'''
For this scenerio, we are comparing countrywide results as a whole so we care more about gender as a population in total,
therefore we will filter out the Sex column for just the "BOTHSEX" value
'''
Total_Disease_df = Diseases_df[(Diseases_df["[Sex]"] == "BOTHSEX")]

In [119]:
#Normallizing the Value, UpperBound, lowerBound columns
Total_Disease_df["Value"] = scaler.fit_transform(Total_Disease_df[["Value"]])
Total_Disease_df["UpperBound"] = scaler.fit_transform(Total_Disease_df[["UpperBound"]])
Total_Disease_df["LowerBound"] = scaler.fit_transform(Total_Disease_df[["LowerBound"]])


#Then we flip the Value, Upper and Lowerbounds that way a high value is a good thing and a low value is the bad
Total_Disease_df.loc["Value"] = 1 - Total_Disease_df["Value"]
Total_Disease_df.loc["UpperBound"] = 1 - Total_Disease_df["UpperBound"]
Total_Disease_df.loc["LowerBound"] = 1 - Total_Disease_df["LowerBound"]

/tmp/ipykernel_21235/1025268515.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Total_Disease_df["Value"] = scaler.fit_transform(Total_Disease_df[["Value"]])
/tmp/ipykernel_21235/1025268515.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Total_Disease_df["UpperBound"] = scaler.fit_transform(Total_Disease_df[["UpperBound"]])
/tmp/ipykernel_21235/1025268515.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value 

In [120]:
Total_Disease_df.head()

,GeoAreaCode,GeoAreaName,TimePeriod,Value,UpperBound,LowerBound,[Nature],[Sex]
2,1.0,World,2000.0,0.406250,0.353612,0.399433,N,BOTHSEX
3,1.0,World,2005.0,0.372396,0.323194,0.373938,N,BOTHSEX
8,1.0,World,2010.0,0.330729,0.290875,0.328612,N,BOTHSEX
11,1.0,World,2015.0,0.299479,0.269962,0.291785,N,BOTHSEX
14,1.0,World,2019.0,0.289062,0.271863,0.271955,N,BOTHSEX


---
---
## SDG 3.5.1: Coverage of Treatment Interventions for Substance Use Disorders.

Tracks the coverage of treatment interventions for substance use disorders.

Specifically, this indicator measures the proportion of people with a drug or alcohol use disorder who have received professional treatment within a given year. The `value` in this dataset is typically expressed as a percentage (%), representing how effectively a country's healthcare system is reaching those who need help with addiction.

Because substance use treatment can vary wildly, this indicator covers several types of interventions:
- Pharmacological treatment: Such as detoxification or maintenance therapy (e.g., methadone).
- Psychosocial services: Including counseling, cognitive behavioral therapy, or structured support groups.
- Rehabilitation: Long-term residential or outpatient programs aimed at social reintegration.

This metric is the primary tool for monitoring progress toward SDG Target 3.5, which aims to "strengthen the prevention and treatment of substance abuse, including narcotic drug abuse and harmful use of alcohol."


In [16]:
#Creating a Dataframe for SDG 3.5.1, Coverage of Treatment Interventions for Substance Use Disorders.
TreatDrug_df = pd.read_csv("SH_SUD_TREAT.csv")
TreatDrug_df.head()

,Goal,Target,Indicator,SeriesCode,SeriesDescription,GeoAreaCode,GeoAreaName,TimePeriod,Value,Time_Detail,...,LowerBound,BasePeriod,Source,GeoInfoUrl,FootNote,[Nature],[Reporting Type],[Sex],[Substance use disorders],[Units]
0,3.0,3.5,3.5.1,SH_SUD_TREAT,Coverage of treatment interventions (pharmacol...,4.0,Afghanistan,2014.0,12.68,2014.0,...,NaN,NaN,Institute for Health Metrics and Evaluation/An...,NaN,NaN,M,G,BOTHSEX,DRUG_TOTAL,PERCENT
1,3.0,3.5,3.5.1,SH_SUD_TREAT,Coverage of treatment interventions (pharmacol...,4.0,Afghanistan,2015.0,24.24,2015.0,...,NaN,NaN,Institute for Health Metrics and Evaluation/An...,NaN,NaN,M,G,BOTHSEX,OPIOIDS,PERCENT
2,3.0,3.5,3.5.1,SH_SUD_TREAT,Coverage of treatment interventions (pharmacol...,4.0,Afghanistan,2015.0,14.23,2015.0,...,NaN,NaN,Institute for Health Metrics and Evaluation/An...,NaN,NaN,M,G,BOTHSEX,DRUG_TOTAL,PERCENT
3,3.0,3.5,3.5.1,SH_SUD_TREAT,Coverage of treatment interventions (pharmacol...,4.0,Afghanistan,2022.0,24.26,2022.0,...,NaN,NaN,Institute for Health Metrics and Evaluation/An...,NaN,NaN,M,G,BOTHSEX,DRUG_TOTAL,PERCENT
4,3.0,3.5,3.5.1,SH_SUD_TREAT,Coverage of treatment interventions (pharmacol...,8.0,Albania,2015.0,17.07,2015.0,...,NaN,NaN,Institute for Health Metrics and Evaluation/An...,NaN,NaN,M,G,BOTHSEX,COCAINE,PERCENT
